# Chapter 1 — Vision Transformer (ViT) from Scratch

**Goal**: Build and understand a Vision Transformer, the image encoder
used in almost all modern multimodal models (CLIP, LLaVA, Flamingo, etc.).

## What you will learn
1. Why images need to be treated as "sequences" for Transformers
2. Patch embedding: splitting an image into tokens
3. Multi-head self-attention (bidirectional — no causal mask)
4. Stacking Transformer encoder blocks
5. The [CLS] token as a global image representation

## Big picture
```
  Image (224×224×3)
       ↓  split into 16×16 patches
  196 patches, each 16×16×3 = 768 values
       ↓  linear projection
  196 patch tokens, each 768-dim
       ↓  prepend [CLS] token
  197 tokens
       ↓  add positional embeddings
       ↓  12 × Transformer encoder block
  [CLS] output → 768-dim image representation
```

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

## 1.1 Patch Embedding

The key insight of ViT: treat image patches like words.

A standard Transformer processes a sequence of vectors.
We create that sequence by dividing the image into fixed-size patches,
then projecting each patch to a vector.

In [ ]:
# Visualise patch splitting
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: original image (fake)
img = np.random.rand(224, 224, 3)
axes[0].imshow(img)
axes[0].set_title('Original Image (224×224)')
axes[0].axis('off')

# Right: overlaid grid showing patches
axes[1].imshow(img)
for i in range(0, 224, 16):
    axes[1].axhline(i, color='red', linewidth=0.5)
    axes[1].axvline(i, color='red', linewidth=0.5)
axes[1].set_title(f'224×224 image split into {(224//16)**2} patches of 16×16')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('figures/ch01_patch_split.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Number of patches: {(224//16)**2}")

In [ ]:
from multimodal_from_scratch.vision.patch_embedding import PatchEmbedding

# Build a PatchEmbedding layer
patch_embed = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, embed_dim=768)

# Test with a dummy batch
batch = torch.randn(4, 3, 224, 224)   # 4 images
out = patch_embed(batch)

print(f"Input shape:  {batch.shape}   (B, C, H, W)")
print(f"Output shape: {out.shape}  (B, n_patches, embed_dim)")
print(f"n_patches = {patch_embed.n_patches}  = (224/16)²")

## 1.2 Multi-Head Self-Attention (Bidirectional)

In Vision Transformers, every patch can attend to every other patch —
there is **no causal mask**.

Compare this to GPT (Chapter 2) where token *i* can only attend to tokens *0..i*.

In [ ]:
from multimodal_from_scratch.vision.attention import MultiHeadSelfAttention

attn = MultiHeadSelfAttention(embed_dim=768, num_heads=12)

# Simulate attending over 197 tokens (196 patches + 1 CLS)
x = torch.randn(2, 197, 768)
out = attn(x)
print(f"Attention input:  {x.shape}")
print(f"Attention output: {out.shape}")

# Count parameters
n_params = sum(p.numel() for p in attn.parameters())
print(f"\nAttention parameters: {n_params:,}")
print(f"  qkv projection:  {768 * 768 * 3:,}  (D → 3D)")
print(f"  out projection:  {768 * 768:,}   (D → D)")

## 1.3 Full ViT Model

In [ ]:
from multimodal_from_scratch.vision.vit import VisionTransformer, ViTConfig

# ViT-Tiny (fast to run on CPU)
cfg = ViTConfig(
    img_size=224,
    patch_size=16,
    embed_dim=192,
    depth=12,
    num_heads=3,
)

vit = VisionTransformer(cfg)
print(f"ViT-Tiny parameters: {vit.count_parameters():,}")

# Forward pass
images = torch.randn(2, 3, 224, 224)
output = vit(images)

print(f"\nInput:              {images.shape}")
print(f"CLS token output:   {output['cls'].shape}  ← global image repr")
print(f"Patch tokens:       {output['tokens'].shape}  ← spatial features")
print(f"All tokens:         {output['all'].shape}  ← cls + patches")

In [ ]:
# Compare model sizes
configs = {
    'ViT-Tiny':  ViTConfig.vit_tiny(),
    'ViT-Small': ViTConfig.vit_small(),
    'ViT-Base':  ViTConfig.vit_base(),
}

print(f"{'Model':<12} {'embed_dim':>10} {'depth':>6} {'heads':>6} {'params':>12}")
print('-' * 50)
for name, cfg in configs.items():
    m = VisionTransformer(cfg)
    print(f"{name:<12} {cfg.embed_dim:>10} {cfg.depth:>6} {cfg.num_heads:>6} "
          f"{m.count_parameters():>12,}")

## Summary

We have built a full Vision Transformer that:
1. Splits an image into 196 patches of 16×16 pixels
2. Projects each patch to a 768-dim embedding
3. Prepends a learnable [CLS] token
4. Adds learnable positional embeddings
5. Processes through 12 bidirectional Transformer blocks
6. Outputs a 768-dim global image representation from [CLS]

In **Chapter 4**, the patch tokens (not just [CLS]) will be fed to the
language decoder so it can "look at" different parts of the image.

**Next**: Chapter 2 — GPT Text Decoder